# Proyección 2027 del mercado de salud animal y del Laboratorio: pipeline completo

Este cuaderno reproduce **de principio a fin** el proceso: lectura sellada de los datos, variables, comparación de modelos con validación cruzada,
y lectura de los resultados de la prueba y de la proyección. Está pensado para ejecutarse con **Kernel > Reiniciar y ejecutar todo**.

**Garantías**
- Semilla fija (`SEMILLA = 0`) y versiones de librerías declaradas en la primera celda (avisa si difieren).
- Rutas **relativas** a la carpeta del cuaderno (`cuaderno/`). Las salidas se escriben en `cuaderno/salida/`, nunca sobre los resultados oficiales.
- **No consulta el examen** (enero a junio de 2026): la evaluación única ya se hizo y el cuaderno solo lee sus resultados guardados.

**Modos** (variable `COMPLETO` en la celda 2): `False` ejecuta una versión reducida (mercado, cuatro modelos, unos minutos);
`True` repite toda la comparación (cuatro tareas, diez modelos, unos 25 minutos) y la compara con los resultados oficiales.

In [1]:
# 1. Entorno, semilla y versiones declaradas
import os, sys, random, platform, warnings
os.environ["PYTHONHASHSEED"] = "0"
SEMILLA = 0
random.seed(SEMILLA)
import numpy as np
np.random.seed(SEMILLA)
import pandas as pd, sklearn, scipy, matplotlib, joblib

VERSIONES_DECLARADAS = {"python": "3.13", "numpy": "2.5", "pandas": "3.0", "scikit-learn": "1.9", "scipy": "1.18", "matplotlib": "3.11", "joblib": "1.6"}
actual = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
          "scipy": scipy.__version__, "matplotlib": matplotlib.__version__, "joblib": joblib.__version__}
for k, v in actual.items():
    ok = v.startswith(VERSIONES_DECLARADAS[k] + ".") or v == VERSIONES_DECLARADAS[k]
    print(f"{k:13s} {v:10s} declarada {VERSIONES_DECLARADAS[k]:6s} {'OK' if ok else 'ADVERTENCIA: version distinta, los resultados pueden variar'}")
    if not ok:
        warnings.warn(f"{k} {v} != {VERSIONES_DECLARADAS[k]}")
print("semilla", SEMILLA)

python        3.13.16    declarada 3.13   OK
numpy         2.5.3      declarada 2.5    OK
pandas        3.0.5      declarada 3.0    OK
scikit-learn  1.9.1      declarada 1.9    OK
scipy         1.18.1     declarada 1.18   OK
matplotlib    3.11.2     declarada 3.11   OK
joblib        1.6.0      declarada 1.6    OK
semilla 0


In [2]:
# 2. Rutas relativas y modo de ejecucion
from pathlib import Path
import json

COMPLETO = True    # True: repite toda la comparacion de modelos (~25 min) y la compara con los resultados oficiales

RAIZ = Path("..") if Path("../src/v2").exists() else Path(".")
assert (RAIZ / "src" / "v2").exists(), "Abra el cuaderno desde la carpeta cuaderno/ o desde la raiz del repositorio"
SALIDA = RAIZ / "cuaderno" / "salida"
os.environ.update({"MF_FUENTE": str(RAIZ / "Base Mercado Salud Animal.csv"), "MF_TRM": str(RAIZ / "TRM.csv"),
                   "MF_DATA": str(SALIDA / "data"), "MF_REP": str(SALIDA / "reports"), "MF_MODELOS": str(SALIDA / "models")})
sys.path.insert(0, str(RAIZ.resolve()))
print("raiz:", RAIZ, "| salida:", SALIDA, "| completo:", COMPLETO)

raiz: .. | salida: ../cuaderno/salida | completo: True


## 3. Lectura y sello de los datos
La base se parte por fecha: **desarrollo** (hasta diciembre de 2025) y **sellado** (desde enero de 2026). Todo lo demás solo lee el archivo de desarrollo.

In [3]:
from src.v2 import config as C, datos, sellar_base, variables
man = sellar_base.sellar()
print(pd.Series(man).to_string())
dev = datos.leer_desarrollo()
print("\nfilas de desarrollo:", len(dev), "| ultimo mes:", dev.Fecha.max().date(), "| series:", dev.groupby(["Especie", "Segmento", "Subgrupo"]).ngroups)

fuente                                       Base Mercado Salud Animal.csv
sha256_fuente            5e0f257d1146ac159f89616588691a7f3b503c4681f9d2...
corte_desarrollo                                                2025-12-01
filas_desarrollo                                                      7632
filas_sellado                                                          636
fecha_max_desarrollo                                            2025-12-01
fecha_min_sellado                                               2026-01-01
sha256_desarrollo        c8a749ca67cfd7b395f1bde79e6d985d115feed6b98a3c...
sha256_sellado           e06ea16b6d8c0cbc45826bbbef914e73f2f3b76f1aaf36...
sellado_el                                             2026-10-21 06:45:40
sha256_trm_fuente        aa8f1fc12e04632495989a481ebb9cdd0d8b0f8984500c...
filas_trm_desarrollo                                                    72
filas_trm_sellado                                                        9
sha256_trm_desarrollo    

## 4. Variables de 12 meses móviles
Crecimiento de las ventas acumuladas de 12 meses (mercado, mercado donde compite el Laboratorio y Laboratorio) y predictores: inflación, PIB, PIB agropecuario, salario mínimo y **precios reales** (cambio del precio promedio de 12 meses, descontada la inflación).

In [4]:
variables.main()
mer = pd.read_csv(C.DATA / "features_desarrollo_mercado.csv", parse_dates=["Fecha"])
mer[["Fecha", "Especie", "y_mercado", "ipc_yoy", "pib", "real_var_cerdo", "real_var_huevo"]].tail(8)

mercado 196 2021-12-01 2025-12-01
compite 196 2021-12-01 2025-12-01
lab 100 2023-12-01 2025-12-01
lab_ym 100 2023-12-01 2025-12-01
series donde compite el Laboratorio: 36


,Fecha,Especie,y_mercado,ipc_yoy,pib,real_var_cerdo,real_var_huevo
188,2025-11-01,CA,0.105779,0.037871,2.525,0.000000,0.000000
189,2025-11-01,Cattle,0.182578,0.037871,2.525,0.000000,0.000000
190,2025-11-01,Poultry,0.088177,0.037871,2.525,0.000000,-0.088016
191,2025-11-01,Swine,0.282493,0.037871,2.525,-0.046950,0.000000
192,2025-12-01,CA,0.101591,0.036360,2.600,0.000000,0.000000
193,2025-12-01,Cattle,0.175965,0.036360,2.600,0.000000,0.000000
194,2025-12-01,Poultry,0.084291,0.036360,2.600,0.000000,-0.079415
195,2025-12-01,Swine,0.263561,0.036360,2.600,-0.073629,0.000000


## 5. Comparación de modelos con validación cruzada por bloques de tiempo y purga
Mismos pliegues y mismo Pipeline (escalamiento incluido) para todos los modelos. Regla de selección declarada: menor error de validación cruzada; entre los que están a menos de un error estándar del mejor, se elige el más simple; debe superar las dos líneas base.

In [5]:
from src.v2 import desarrollo, nucleo as N
if not COMPLETO:
    # version reducida: solo la tarea "mercado" y cuatro modelos con rejillas pequenas (demostracion; no reemplaza los resultados oficiales)
    for k in list(N.EST):
        if k not in ("ridge", "enet", "knn", "gbm"):
            del N.EST[k]
    N.GRID["gbm"] = {"n_estimators": [50, 100], "learning_rate": [0.03], "max_depth": [2, 3], "min_samples_leaf": [8]}
    N.GRID["knn"] = {"n_neighbors": [12, 20], "weights": ["uniform", "distance"], "p": [2]}
    tareas = ["mercado"]
else:
    tareas = list(N.TASKS)
C.REP.mkdir(parents=True, exist_ok=True)
for t in tareas:
    desarrollo.dev(t)


=== mercado: 196 filas de entrenamiento, 9 pliegues, embargo 6 meses
    clase  n  pct
  regular  9  4.6
  estable 95 48.5
optimista 92 46.9


  ols_esp    MAE_cv=14.87 ± 8.86  {} balance=False


  ridge_esp  MAE_cv= 4.89 ± 1.32  {'alpha': 1000} balance=False


  enet_esp   MAE_cv= 5.03 ± 1.49  {'alpha': 0.1, 'l1_ratio': 0.5} balance=False


  ols        MAE_cv=12.95 ± 9.21  {} balance=False


...

  bagging    MAE_cv= 5.90 ± 1.24  {'estimator__max_depth': None, 'estimator__min_samples_leaf': 3, 'max_samples': 1.0} balance=False


  rf         MAE_cv= 5.79 ± 1.16  {'max_depth': 8, 'max_features': 0.5, 'min_samples_leaf': 3} balance=True


  gbm        MAE_cv= 5.28 ± 1.83  {'learning_rate': 0.03, 'max_depth': 3, 'min_samples_leaf': 3, 'n_estimators': 400} balance=True
                 modelo  MAE_cv   sd  MAE_oof_global  MAE_oof_subconjunto_lag  R2_oof  R2_entrenamiento
                ols_esp    9.62 3.89            9.62                    11.18   -0.29              0.95
              ridge_esp    5.45 1.96            5.45                     4.92    0.65              0.91
               enet_esp    5.66 2.63            5.66                     4.77    0.60              0.90
                    ols    7.52 2.53            7.52                     6.86    0.24              0.91
                  ridge    6.83 2.66            6.83                     6.51    0.43              0.91
                   enet    6.60 2.50            6.60                     6.01    0.48              0.91
                    knn    5.99 2.00            5.99                     5.33    0.60              1.00
                bagging    5.90 1.24 

 meses_entrenamiento  filas  MAE_entrenamiento  MAE_retenido
                   4     16               1.30          9.51
                   8     32               1.97          5.83
                  12     48               2.81          8.18
                  16     64               2.64          6.04


  umbral recomendado: restar m = -0.5 pp antes de clasificar


In [6]:
# Resultados de la comparacion (MAE en puntos porcentuales y R2 fuera de pliegue)
for t in tareas:
    print("\n==", t)
    display(pd.read_csv(C.REP / f"comparacion_cv_{t}.csv").round(2))
    s = json.loads((C.REP / f"seleccion_{t}.json").read_text())
    print("modelo elegido:", s["nombre"], s["params"], "| MAE de CV:", round(s["MAE_cv_final"], 2), "| supera las bases:", s["supera_lineas_base"])


== mercado


,modelo,MAE_cv,sd,MAE_oof_global,MAE_oof_subconjunto_lag,R2_oof,R2_entrenamiento
0,ols_esp,14.87,8.86,14.82,13.83,-11.92,0.85
1,ridge_esp,4.89,1.32,4.91,4.57,-0.11,0.36
2,enet_esp,5.03,1.49,5.05,4.62,-0.15,0.34
3,ols,12.95,9.21,12.60,13.69,-6.36,0.64
4,ridge,4.62,2.06,4.62,4.00,-0.15,-0.03
5,enet,4.61,1.69,4.57,4.19,-0.03,0.00
6,knn,4.31,1.28,4.32,4.01,0.06,0.77
7,bagging,4.58,1.28,4.66,4.26,0.02,0.73
8,rf,4.52,1.59,4.57,4.12,-0.03,0.50
9,gbm,4.17,1.48,4.24,3.67,0.11,0.69


modelo elegido: Elastic Net común {'alpha': 0.1, 'l1_ratio': 0.5} | MAE de CV: 4.61 | supera las bases: True

== compite


,modelo,MAE_cv,sd,MAE_oof_global,MAE_oof_subconjunto_lag,R2_oof,R2_entrenamiento
0,ols_esp,12.29,6.77,12.17,10.89,-3.88,0.90
1,ridge_esp,6.42,2.57,6.42,5.92,-0.16,0.36
2,enet_esp,6.60,2.55,6.60,6.13,-0.21,0.27
3,ols,12.49,6.95,12.43,12.28,-2.92,0.63
4,ridge,5.82,2.25,5.81,5.40,-0.11,-0.09
5,enet,5.89,2.26,5.89,5.43,-0.12,-0.11
6,knn,6.25,1.49,6.24,6.02,-0.05,1.00
7,bagging,5.60,1.41,5.64,5.21,0.12,0.82
8,rf,5.71,1.72,5.71,5.25,0.11,0.70
9,gbm,5.46,2.27,5.47,4.87,-0.01,0.39


modelo elegido: Ridge común {'alpha': 10000} | MAE de CV: 5.82 | supera las bases: True

== lab


,modelo,MAE_cv,sd,MAE_oof_global,MAE_oof_subconjunto_lag,R2_oof,R2_entrenamiento
0,ols_esp,12.07,5.53,12.07,13.69,-1.04,0.94
1,ridge_esp,5.52,1.98,5.52,4.85,0.64,0.91
2,enet_esp,5.75,2.59,5.75,4.73,0.59,0.90
3,ols,7.93,2.18,7.93,7.25,0.19,0.91
4,ridge,7.37,2.48,7.37,7.01,0.36,0.91
5,enet,7.38,2.38,7.38,6.98,0.35,0.91
6,knn,6.02,1.93,6.02,5.35,0.60,1.00
7,bagging,6.02,1.37,6.02,5.59,0.62,0.96
8,rf,5.93,1.17,5.93,5.34,0.64,0.94
9,gbm,5.71,2.24,5.71,5.34,0.61,1.00


modelo elegido: Ridge por especie {'alpha': 10} | MAE de CV: 5.52 | supera las bases: True

== lab_ym


,modelo,MAE_cv,sd,MAE_oof_global,MAE_oof_subconjunto_lag,R2_oof,R2_entrenamiento
0,ols_esp,9.62,3.89,9.62,11.18,-0.29,0.95
1,ridge_esp,5.45,1.96,5.45,4.92,0.65,0.91
2,enet_esp,5.66,2.63,5.66,4.77,0.60,0.90
3,ols,7.52,2.53,7.52,6.86,0.24,0.91
4,ridge,6.83,2.66,6.83,6.51,0.43,0.91
5,enet,6.60,2.50,6.60,6.01,0.48,0.91
6,knn,5.99,2.00,5.99,5.33,0.60,1.00
7,bagging,5.90,1.24,5.90,5.52,0.62,0.97
8,rf,5.79,1.16,5.79,5.25,0.65,0.95
9,gbm,5.28,1.83,5.28,4.45,0.66,1.00


modelo elegido: Ridge por especie {'alpha': 10} | MAE de CV: 5.45 | supera las bases: True


In [7]:
# Reproducibilidad: en modo completo, la seleccion debe coincidir con la oficial (reports/v2)
if COMPLETO:
    for t in tareas:
        a = json.loads((C.REP / f"seleccion_{t}.json").read_text()); b = json.loads((RAIZ / "reports" / "v2" / f"seleccion_{t}.json").read_text())
        print(t, "identica a la oficial:", a == b)
else:
    print("Modo reducido: la comparacion con los resultados oficiales solo aplica con COMPLETO = True.")

mercado identica a la oficial: True
compite identica a la oficial: True
lab identica a la oficial: True
lab_ym identica a la oficial: True


## 6. Examen de 2026 (una sola consulta) y proyección 2027
El cuaderno **no** consulta el examen: lee los resultados guardados cuando se hizo la evaluación única. Si el candado no existe, la evaluación aún no se ha hecho y se corre con `python -m src.v2.prueba`.

In [8]:
OFICIAL = RAIZ / "reports" / "v2"
lock = OFICIAL / "prueba_consultada.lock"
if lock.exists():
    print("El examen se consulto una sola vez el", lock.read_text().strip())
    r = json.loads((OFICIAL / "prueba_resultados.json").read_text())
    filas = []
    for t in ("mercado", "compite", r["variante_lab"]):
        x = r[t]
        filas.append({"tarea": t, "modelo": x["modelo_final"], "MAE_pp": x["MAE_pp"], "base promedio especie": x["B2_media_especie"]["MAE_pp"],
                      "base año anterior": x["B1_crecimiento_hace_12m"]["MAE_pp"], "R2": x["R2"]})
    display(pd.DataFrame(filas).round(2))
else:
    print("El examen aun no se ha consultado. No lo ejecute desde este cuaderno.")

El examen se consulto una sola vez el 2026-10-21 06:41:48


,tarea,modelo,MAE_pp,base promedio especie,base año anterior,R2
0,mercado,Elastic Net común,9.24,8.66,8.39,-0.02
1,compite,Ridge común,11.82,11.72,11.18,-0.03
2,lab_ym,Ridge por especie,4.43,4.08,6.28,0.43


In [9]:
proy = OFICIAL / "proyeccion_2027_base.csv"
if proy.exists():
    display(pd.read_csv(proy).round(3))
else:
    print("La proyeccion final se genera despues de la evaluacion unica.")

,Especie,crec_mercado,crec_lab,escenario_mercado,escenario_lab
0,CA,0.154,0.216,optimista,optimista
1,Cattle,0.154,0.049,optimista,regular
2,Poultry,0.154,0.149,optimista,estable
3,Swine,0.154,0.257,optimista,optimista
4,Total,0.154,0.206,optimista,optimista


## 7. Verificación del sello
Resultado guardado de la prueba del canario (cifras de 2026 reemplazadas por un valor absurdo: el desarrollo debe dar resultados idénticos).

In [10]:
ver = OFICIAL / "verificacion_sello.txt"
print(ver.read_text() if ver.exists() else "Sin registro de verificacion.")
print("accesos a la prueba durante el desarrollo:", (OFICIAL / "accesos_prueba.log").read_text() if (OFICIAL / "accesos_prueba.log").exists() else "ninguno registrado en esta carpeta")

estatica_ok=True
canario_identicos=48/48
accesos_prueba=0

accesos a la prueba durante el desarrollo: 2026-10-21 06:41:47	evaluacion unica de prueba	/home/user/MercadoFarma/src/v2/prueba.py
2026-10-21 06:42:40	reajuste final (posterior a la evaluacion unica)	/home/user/MercadoFarma/src/v2/despliegue.py
2026-10-21 06:42:43	proyeccion 2027 (posterior a la evaluacion unica)	/home/user/MercadoFarma/src/v2/proyeccion.py

